# CovCollExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/CovCollExamples.md`


Notebook source link: [CovCollExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/CovCollExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "CovCollExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"CovCollExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"CovCollExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"CovCollExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"CovCollExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "load CovariateSample.mat;",
    "cc=CovColl({position,force});",
    "figure; cc.plot; %plots all covariates and their components",
    "cc.getCov(1); %returns position;",
    "cc.getCov('Position');",
    "cc.getCov({'Position','Force'});",
    "cc.resample(200); %resamples both position and force",
    "cc.setMask({{'Position','x'},{'Force','f_y'}});",
    "figure; cc.plot; %plot only x and f_y;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for CovCollExamples.")


In [ ]:
# CovCollExamples: covariate collection queries, masking, and resampling.
from nstat.compat.matlab import Covariate, CovColl, History, nspikeTrain

t = np.arange(0.0, 5.0 + 0.001, 0.001)
position = Covariate(
    time=t,
    data=np.column_stack([np.exp(-t), np.sin(2.0 * np.pi * t), np.sin(2.0 * np.pi * t) ** 3]),
    name="Position",
    labels=["x", "y", "z"],
)
force = Covariate(
    time=t,
    data=np.column_stack([np.abs(np.sin(2.0 * np.pi * t)), np.abs(np.sin(2.0 * np.pi * t)) ** 2]),
    name="Force",
    labels=["f_x", "f_y"],
)
cc = CovColl([position, force])

fig1 = plt.figure(figsize=(9.0, 4.2))
cc.plot()
plt.title(f"{TOPIC}: all covariates")
plt.xlabel("time [s]")
plt.tight_layout()
plt.show()

_pos = cc.getCov("Position")
_force = cc.getCov("Force")
cc.resample(200.0)
cc.setMask(["Position", "Force"])

fig2 = plt.figure(figsize=(9.0, 4.2))
cc.plot()
plt.title("Resampled/masked covariates")
plt.xlabel("time [s]")
plt.tight_layout()
plt.show()

X, labels = cc.dataToMatrix()
n_before_remove = cc.nActCovar()
cc.removeCovariate("Force")
n_after_remove = cc.nActCovar()

assert X.shape[1] >= 4
assert n_after_remove == max(1, n_before_remove - 1)
history = History(bin_edges_s=np.array([0.0, 0.01, 0.03], dtype=float))
spikes = nspikeTrain(spike_times=np.sort(rng.random(25) * 0.5), t_start=0.0, t_end=0.5, name="tmp")
H = history.computeHistory(spikes.spike_times, np.arange(0.0, 0.5, 0.01))
assert H.ndim == 2 and H.shape[1] == history.n_bins
assert spikes.spike_times.size > 5

CHECKPOINT_METRICS = {
    "matrix_rows": float(X.shape[0]),
    "matrix_cols": float(X.shape[1]),
    "active_covariates_after_remove": float(n_after_remove),
}
CHECKPOINT_LIMITS = {
    "matrix_rows": (200.0, 2000.0),
    "matrix_cols": (4.0, 8.0),
    "active_covariates_after_remove": (1.0, 3.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
